# Mask2Former — Intro Semantic Segmentation Notebook

This notebook continues the `cv-research-lab` autonomous vehicle perception track.

## Goals

- Run Mask2Former on a street-scene image
- Generate a semantic segmentation map
- Compare semantic segmentation with SAM-style mask proposal generation
- Measure inference time and approximate GPU memory usage
- Document observations in the context of autonomous vehicle perception systems

## Why Mask2Former?

SAM can generate strong region masks, but those masks are not inherently semantic. In autonomous vehicle perception, it is not enough to know *where* regions are; the system also needs to know *what* they are, such as road, vehicle, pedestrian, sidewalk, traffic sign, or building.

Mask2Former is useful to explore because it connects modern transformer-based segmentation with semantic scene understanding.


## 1. Runtime Setup

In Colab, go to:

`Runtime` → `Change runtime type` → `Hardware accelerator` → `GPU`

Then run the cell below.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
device


## 2. Install Dependencies

This notebook uses Hugging Face Transformers to load a pretrained Mask2Former model.


In [ ]:
!pip -q install transformers accelerate pillow matplotlib opencv-python


## 3. Imports

In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation


## 4. Load a Street-Scene Image

This example uses a public street image containing vehicles and urban scene structure.

Later, replace this with a dashcam image, a KITTI frame, a nuScenes frame, or your own driving-scene image.


In [ ]:
# Public street-scene image used for object detection demos
!wget -q -O street_scene.jpg https://raw.githubusercontent.com/ultralytics/assets/main/bus.jpg

image = Image.open("street_scene.jpg").convert("RGB")
image_np = np.array(image)

plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.title("Input Street Scene")
plt.axis("off")
plt.show()

print("Image size:", image.size)
print("Image array shape:", image_np.shape)


## 5. Load Mask2Former

This notebook uses a Cityscapes-trained Mask2Former checkpoint.

Cityscapes is a street-scene dataset, making it more relevant for autonomous vehicle perception than generic object segmentation.

The model produces semantic segmentation labels such as road, sidewalk, building, person, rider, car, bus, truck, bicycle, and more.


In [ ]:
model_id = "facebook/mask2former-swin-small-cityscapes-semantic"

processor = AutoImageProcessor.from_pretrained(model_id)
model = Mask2FormerForUniversalSegmentation.from_pretrained(model_id)

model.to(device)
model.eval()

id2label = model.config.id2label

print("Loaded:", model_id)
print("Number of labels:", len(id2label))
print(id2label)


## 6. Run Semantic Segmentation Inference

This cell measures the time required to run Mask2Former on one image.

For autonomous vehicles, inference speed is a major constraint because perception systems must process continuous video streams in real time.


In [ ]:
if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

inputs = processor(images=image, return_tensors="pt").to(device)

start = time.time()

with torch.no_grad():
    outputs = model(**inputs)

if device == "cuda":
    torch.cuda.synchronize()

end = time.time()

target_sizes = [image.size[::-1]]  # PIL gives (width, height); model expects (height, width)
predicted_segmentation = processor.post_process_semantic_segmentation(
    outputs,
    target_sizes=target_sizes
)[0]

inference_time = end - start

print(f"Inference time: {inference_time:.3f} seconds")
print("Segmentation map shape:", tuple(predicted_segmentation.shape))

if device == "cuda":
    peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
    print(f"Peak GPU memory allocated: {peak_memory_mb:.1f} MB")


## 7. Inspect Predicted Classes

The segmentation map assigns one class ID to each pixel.

This is a key difference from SAM automatic mask generation:

- SAM proposes unlabeled masks
- Mask2Former predicts semantic labels for every pixel


In [ ]:
seg = predicted_segmentation.cpu().numpy()

unique_class_ids = np.unique(seg)

print("Predicted class IDs:", unique_class_ids)
print("\nPredicted classes:")

for class_id in unique_class_ids:
    label = id2label.get(int(class_id), f"class_{class_id}")
    pixel_count = int((seg == class_id).sum())
    print(f"{class_id:02d}: {label:<15} pixels={pixel_count}")


## 8. Visualize Semantic Segmentation

The visualization below creates a colored segmentation map and blends it with the original image.

Each color corresponds to a semantic class.


In [ ]:
def create_color_map(num_classes, seed=42):
    """Create deterministic colors for each class."""
    rng = np.random.default_rng(seed)
    colors = rng.integers(0, 255, size=(num_classes, 3), dtype=np.uint8)
    return colors


num_classes = max(max(id2label.keys()) + 1, int(seg.max()) + 1)
colors = create_color_map(num_classes)

seg_color = colors[seg]

alpha = 0.55
overlay = (alpha * seg_color + (1 - alpha) * image_np).astype(np.uint8)

plt.figure(figsize=(12, 8))
plt.imshow(overlay)
plt.title("Mask2Former Semantic Segmentation Overlay")
plt.axis("off")
plt.show()


## 9. Display Segmentation Map Alone

This removes the original image and shows only the predicted semantic regions.


In [ ]:
plt.figure(figsize=(12, 8))
plt.imshow(seg_color)
plt.title("Predicted Semantic Segmentation Map")
plt.axis("off")
plt.show()


## 10. Create a Legend for Predicted Classes

This legend shows only the classes that appear in the current image.


In [ ]:
import matplotlib.patches as mpatches

patches = []

for class_id in unique_class_ids:
    label = id2label.get(int(class_id), f"class_{class_id}")
    color = colors[int(class_id)] / 255.0
    patches.append(mpatches.Patch(color=color, label=f"{class_id}: {label}"))

plt.figure(figsize=(8, max(2, len(patches) * 0.35)))
plt.legend(handles=patches, loc="center left", frameon=False)
plt.axis("off")
plt.title("Predicted Class Legend")
plt.show()


## 11. Compare With SAM Automatic Mask Generation

Use this section to write observations comparing the Mask2Former output to your earlier SAM notebook.

### Conceptual Comparison

| Model | Output | Main Strength | Limitation |
|---|---|---|---|
| SAM automatic mask generation | Many unlabeled candidate masks | Strong generic region proposals | Does not assign semantic class labels |
| Mask2Former semantic segmentation | One class label per pixel | Scene understanding through semantic labels | May miss fine object boundaries or small objects |

### Questions

- Which output is more useful for autonomous vehicle perception?
- Which model better supports semantic understanding?
- Which model better captures fine object boundaries?
- Which model is more appropriate for real-time deployment?
- What additional components would be needed for tracking and prediction?


## 12. Autonomous Vehicle Perception Notes

Autonomous vehicle perception systems additionally require:

- Processing continuous video streams across time
- Low-latency, real-time inference
- Object tracking and trajectory prediction
- Semantic understanding of regions, not just region boundaries
- Integration across a full perception stack, including multi-camera and multi-sensor fusion
- Operation under limited onboard compute, memory, and power constraints
- High levels of safety, robustness, and reliability


### Observations

Write your initial observations here.

Suggested prompts:

- Which classes were segmented correctly?
- Which objects or regions were missed?
- Were small or distant objects handled well?
- Were boundaries clean or coarse?
- Did the model distinguish vehicles, people, road, sidewalk, and buildings?
- How does this differ from SAM's automatic mask generation?
- Was inference speed plausible for real-time perception?

Example starter observation:

> Mask2Former provides semantic labels for each pixel, making its output more directly useful for autonomous vehicle perception than unlabeled SAM masks. However, the model still needs to be evaluated for latency, small-object performance, boundary quality, and temporal consistency across video frames.


## 13. Next Experiments

Possible next steps:

1. Run this notebook on several different street scenes.
2. Compare Mask2Former against SAM on the same image.
3. Measure inference time across image sizes.
4. Try a larger or smaller Mask2Former checkpoint.
5. Test on images with pedestrians, cyclists, traffic signs, and distant vehicles.
6. Explore video segmentation or tracking-by-detection.
7. Move from single-frame perception to temporal perception.
